# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shoriful-mynul/flyrank-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


**Lane:** Content Refresh Prioritization

**Decision:** Which existing content pages should be refreshed first?

**Action:** A content manager reviews the ranked list and updates the highest-priority pages.

**Wrong-call cost:** Refreshing low-priority pages instead of high-impact pages can waste time and resources while important pages continue losing traffic and engagement.

## 0. Setup and data access

This notebook uses the gated FlyRank warehouse release hosted on Hugging Face.

The Hugging Face read token is stored in Colab Secrets as `HF_TOKEN` and is never written directly into the notebook.

In [35]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [36]:
%pip -q install duckdb huggingface_hub

In [37]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

In [38]:
TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("Warehouse connection configured.")

Warehouse connection configured.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data contract

**1. What one row means:**  
One row represents one content item for one client on one report date.

**2. Tables used:**  
The main table is `fact_content_daily_performance`, joined with `dim_content` or `dim_clients` only when content or client context is needed.

**3. Time window:**  
For this development slice, I use the March 2026 partition (`month=2026-03`). The final modeling workflow should use features that are available before the decision moment and should keep the outcome window separate from the feature window.

**4. What is predicted/ranked:**  
The outcome proxy is future search performance, represented by March 2026 Google Search clicks (`mar_clicks`). The goal is to use February information to prioritize content for refresh review.

**5. One thing deliberately excluded:**  
March outcome fields, including `mar_clicks`, are excluded from the legitimate feature set because they would not be known at the February decision moment. `trend_direction` and `trend_pct` are also excluded because they are closely related to decline labels and could leak outcome information.

## 2. Fields: feature / label / context / excluded
*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification

**Label / proxy**

`is_declining_label` is the provisional target. It represents whether the content item is currently labelled as declining.

**Context**

`client_hash_id` and `content_hash_id` are identifiers used for grouping, joining, and splitting. They are not model features.

**Excluded**

`trend_direction` and `trend_pct` are excluded because they are directly related to the construction of the provisional declining label and could leak target information.

**Candidate features**

1. February Google Search impressions
2. February Google Search clicks
3. February average search position
4. February GA4 sessions
5. February scroll events

These five features are calculated from February 2026 and are intended to represent information available before the March outcome window.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Query 1 — Grain

The expected grain is one row per `report_date × client_hash_id × content_hash_id`.

The query below checks whether any combination appears more than once.

In [39]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {TABLES["fact_daily"]}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


### Verification Query 2 — March 2026 slice

This query verifies the number of rows in the March 2026 development slice and confirms its date range.

In [40]:
slice_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM {TABLES["fact_daily"]}
""").df()

slice_check

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


### Verification Query 3 — Data availability

Some clients do not have usable GA4 history for every row. The availability flag must therefore be checked explicitly rather than treating zero-valued metrics as evidence of availability.

The query uses `IS TRUE` to retain only rows where GA4 data is explicitly available.

In [41]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS available_rows,
        MIN(report_date) AS first_available_date,
        MAX(report_date) AS last_available_date
    FROM {TABLES["fact_daily"]}
    WHERE ga4_data_available IS TRUE
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows,first_available_date,last_available_date
0,413966,2026-03-01,2026-03-31


### Five-feature frame

I use February 2026 as the feature window so that every feature represents information available before the March outcome window.

The five features are:

1. **February Google Search impressions:** shows how much search visibility the content received before the decision point.
2. **February Google Search clicks:** shows the amount of search traffic generated before the decision point.
3. **February average search position:** summarizes how prominently the content appeared in search results before the decision point.
4. **February GA4 sessions:** shows the amount of measured site traffic received before the decision point.
5. **February scroll events:** provides a historical engagement signal that was already observable before the decision point.

All five features are calculated from the February window only. They are therefore intended to represent information that could have been known before evaluating the March outcome.

In [42]:
FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("FEB:", FEB)
print("MAR:", MAR)

FEB: read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')
MAR: read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')


In [43]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS feb_impressions,

        SUM(gsc_clicks) AS feb_clicks,

        SUM(gsc_sum_position) /
            NULLIF(SUM(gsc_impressions), 0) AS feb_avg_position,

        SUM(ga4_sessions) AS feb_sessions,

        SUM(scroll_events) AS feb_scroll_events

    FROM {FEB}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)

display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (153559, 7)


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position,feb_sessions,feb_scroll_events
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.448161,0.0,0.0
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.316508,6.0,0.0
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,9.966926,1.0,0.0
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,41.814739,6.0,1.0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,10.307216,3.0,1.0


## Leakage trap

The legitimate features use only February information, while the outcome comes from March.

To demonstrate leakage, I will intentionally add a March outcome field to the feature set. This information would not have been available when making the refresh-prioritization decision.

If the model score becomes unrealistically strong after adding this future outcome, that demonstrates target leakage. I will then remove the leaked field and keep only the features that were available before the outcome window.

In [44]:
label_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE NULL
            END
        ) AS mar_clicks,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE NULL
            END
        ) AS mar_impressions,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS measured_days

    FROM {MAR}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Label frame shape:", label_frame.shape)

display(label_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Label frame shape: (331437, 5)


,client_hash_id,content_hash_id,mar_clicks,mar_impressions,measured_days
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.0,6523.0,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,0.0,453.0,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,6.0,5630.0,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,13.0,4944.0,31
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,0.0,42.0,21


In [45]:
model_df = feature_frame.merge(
    label_frame,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Model frame shape:", model_df.shape)

display(model_df.head())

Model frame shape: (145279, 10)


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position,feb_sessions,feb_scroll_events,mar_clicks,mar_impressions,measured_days
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.448161,0.0,0.0,0.0,315.0,29
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.316508,6.0,0.0,4.0,14536.0,29
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,9.966926,1.0,0.0,0.0,387.0,29
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,41.814739,6.0,1.0,5.0,4697.0,29
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,10.307216,3.0,1.0,1.0,1004.0,29


### Honest baseline

The honest model uses only the five February features to estimate the March click outcome.

The March outcome is kept separate from the inputs because it represents information that would only become available after the decision window.

In [46]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

honest_features = [
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position",
    "feb_sessions",
    "feb_scroll_events"
]

honest_df = model_df.dropna(
    subset=honest_features + ["mar_clicks"]
).copy()

X = honest_df[honest_features]
y = honest_df["mar_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

honest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)

honest_r2 = r2_score(
    y_test,
    honest_predictions
)

print("Honest model R²:", round(honest_r2, 4))

Honest model R²: 0.6767


### Deliberate leakage experiment

For this experiment only, I intentionally add `mar_clicks` itself as an input feature.

This is invalid in a real decision-support system because March clicks are the outcome we are trying to estimate. At the decision moment, March clicks would not yet be known.

The purpose is to show how using future outcome information can create an artificially strong model score.

In [47]:
leaky_features = honest_features + ["mar_clicks"]

leaky_df = model_df.dropna(
    subset=leaky_features
).copy()

X_leak = leaky_df[leaky_features]
y_leak = leaky_df["mar_clicks"]

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leak,
    y_leak,
    test_size=0.30,
    random_state=42
)

leaky_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train_leak, y_train_leak)

leaky_predictions = leaky_model.predict(X_test_leak)

leaky_r2 = r2_score(
    y_test_leak,
    leaky_predictions
)

print("Leaky model R²:", round(leaky_r2, 4))

Leaky model R²: 0.9991


### Final honest feature set

The leaked outcome field is removed. The final feature set contains only information available during the February decision window.

This is the version that should be retained for future modeling work.

In [48]:
final_features = [
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position",
    "feb_sessions",
    "feb_scroll_events"
]

final_df = model_df.dropna(
    subset=final_features + ["mar_clicks"]
).copy()

X_final = final_df[final_features]
y_final = final_df["mar_clicks"]

X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_final,
    y_final,
    test_size=0.30,
    random_state=42
)

final_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train_final, y_train_final)

final_predictions = final_model.predict(X_test_final)

final_r2 = r2_score(
    y_test_final,
    final_predictions
)

print("Final honest R²:", round(final_r2, 4))

print("\nFinal features:")
for feature in final_features:
    print("-", feature)

Final honest R²: 0.6767

Final features:
- feb_impressions
- feb_clicks
- feb_avg_position
- feb_sessions
- feb_scroll_events


### Leakage result

The deliberate leakage experiment demonstrates why future outcome information must not be included as a feature.

The honest model uses only February information, while the leaked model receives `mar_clicks`, which is the March outcome itself. The stronger leaked score is therefore not evidence of a better model; it is evidence that the model has been given information that would not exist at the decision moment.

For the final feature set, the leaked field is removed and only the five February features are retained.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The main limitation is that data availability is not uniform across all client-content records. Search and analytics data can be unavailable for some observations, so missing data should not automatically be interpreted as zero performance.

March clicks are used only as a future performance proxy. They do not prove that refreshing a page will cause clicks or rankings to increase.

The feature frame also represents historical signals rather than a complete description of content quality, competition, or editorial effort. Therefore, the output should be treated as decision support for human review rather than an automatic refresh decision.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] Exactly three verification queries are shown with outputs.
- [x] Data availability is checked explicitly with `IS TRUE`.
- [x] The unit of analysis is represented by a real dataframe.
- [x] Exactly five decision-time features are defined.
- [x] A future outcome is deliberately introduced as a leakage experiment.